# Download of PM10 data from API

Both hourly and daily data

In [ ]:
import asyncio
from pathlib import Path
import nest_asyncio
import pandas as pd

from airbase import AirbaseClient
from airbase.parquet_api import download, Dataset, AggregationType

In [ ]:
DOWNLOAD_DIR = Path("data/pm10_daily")
DOWNLOAD_DIR2 = Path("data/pm10_hourly")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR2.mkdir(parents=True, exist_ok=True)

In [ ]:
client = AirbaseClient()

In [ ]:
await download(
    dataset=Dataset.Verified,
    root_path=DOWNLOAD_DIR2,
    countries=frozenset(client.countries),
    pollutants={"PM10"},
    frequency=AggregationType.Hourly,
    country_subdir=True,
    overwrite=False,
    quiet=False,
)

In [ ]:
await download(
    dataset=Dataset.Verified,
    root_path=DOWNLOAD_DIR,
    countries=frozenset(client.countries),
    pollutants={"PM10"},
    frequency=AggregationType.Daily,
    country_subdir=True,
    overwrite=False,
    quiet=False,
)

In [ ]:
nest_asyncio.apply()  # Allows nested asyncio loops inside Jupyter

# Now you can call client directly without error:
client.download_metadata(DOWNLOAD_DIR / "metadata.csv")

## Read all daily data and save them as .parquet and .csv

In [ ]:
# Read all downloaded parquet files at once
df = pd.read_parquet(DOWNLOAD_DIR)

if "Start" in df.columns:
    df["Start"] = pd.to_datetime(df["Start"])
    pm10_2024 = df[(df["Start"] >= "2024-01-01") & (df["Start"] < "2025-01-01")]
else:
    pm10_2024 = df

pm10_2024.to_parquet("pm10_europe_daily_2024.parquet", index=False)
pm10_2024.to_csv("pm10_europe_daily_2024.csv", index=False)